В основном вся нормативная документация выгружена из КонсультанПлюс и в каждом документе присутствует структура.
Виды нормативной документации:
1.	Федеральный закон
2.	ГОСТ
3.	Постановление Правительства РФ
4.	Приказ Министерства промышленности и торговли
5. и т.д.

In [ ]:
#установка библиотеки docx

In [1]:
pip install python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.6/239.6 kB 1.8 MB/s eta 0:00:00


In [2]:
import re
import requests
import os
import io
import tempfile
from docx import Document
from docx.oxml.ns import nsdecls
from docx.oxml import parse_xml

In [ ]:
#Код для Apps Script для получения списка файлов и их ID
'''
function listFilesInFolder() {
  var folder = DriveApp.getFolderById('1kldI9-mNiesJ4Tj7MQyOiLyzfO0ujVzK');
  var files = folder.getFiles();

  // Создаем новый txt-файл
  var file = DriveApp.createFile('FileList.txt', '');

  // Получаем доступ к содержимому файла
  var fileContent = file.getBlob().getDataAsString();

  while (files.hasNext()) {
    var fileInFolder = files.next();
    // Записываем информацию о файле в txt-файл
    fileContent += fileInFolder.getName() + ': ' + fileInFolder.getId() + '\n';
  }

  // Записываем обновленное содержимое обратно в файл
  file.setContent(fileContent);
}
'''

In [3]:
# Перечень нормативной документации сохранен в файле FileList.txt, его ID 1xlBmSKGs1Wi5ti5xjgKGEIewggvknRxH
id_FileList_txt = '1xlBmSKGs1Wi5ti5xjgKGEIewggvknRxH'

In [4]:
# функция для загрузки документа по doc_id из гугл драйв в текстовом формате
def load_doc_id_text(file_id):
    # Download the document as plain text
    response = requests.get(f'https://drive.google.com/uc?export=download&id={file_id}')
    response.raise_for_status()
    text = response.text

    return text

In [5]:
# Выгрузим перечень нормативной документации
data_txt_id= load_doc_id_text(id_FileList_txt)
data_txt_id

'Постановление 1847.docx: 16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh\nСОГЛАШЕНИЕ Бурабай 2015.docx: 1_u0BBIJphrq2qBsqog5H0B9OgZjWO2bt\n162-ФЗ.docx: 1zQ_Xn771EuWl9-VZDAiQy2IThdpdgLys\nПриказ МПТ 971.docx: 15J5AZZvjRe_vCYze44tB-fk3amL-2YM0\nПриказ РСТ 2173.docx: 1r4wUVjTqWtNyB-rmI76DEsimSyBIqGlF\nПриказ РСТ 2346.docx: 1nJdos1CgVTcbtQtFhorMoxFPBx6BBgoe\nПостановление 521.docx: 1F17SNGzRYBiHNVLbB6DK6RlVx5LSbeRN\nПМГ 06.2019.docx: 1CxtJO4dVKybY1swOkcXP4buJt1MYoBgb\nРМГ 29.2013.docx: 1qHfddurJ1QEc78sdAeHlVjBL3pflmnbJ\nПриказ МПТ 2907.docx: 1b30hYkfRiqO8TJVZpBwn5ui_RypiJxk0\nПриказ МПТ 456.docx: 1y_2d-GSEctG2q52Pu46fZ-L0cBppAL_y\nПриказ МПТ 4091.docx: 1R5pq1BhN59DXgx2MK-7sH0QbzJN_opWm\nПриказ МПТ 2167.docx: 1zuX2wQqoGKKl5eF9m1PuDTEqnQj9Z-1p\nПриказ МПТ 2906.docx: 1LCNEKwyZDQujUJgMHFdXL9x_kdbkwxoj\nПостановление 734.docx: 1qCFfa--cQb4crDu2uSCytUQcxsx7LlyJ\nПостановление 1053.docx: 1C0lbveUI49b9DVJWzaeTXvTkRFpyssfn\nПостановление 879.docx: 1S1oxQAb98MZJH-zb5JNMbDgoDlgjlLqA\nПостановление 311.docx: 1opVc

In [6]:
#Cохраняем название файлов, данные ID в отдельный библиотеку python
# Создаем пустой словарь
file_dict = {}

# Используем регулярные выражения для поиска всех файлов и их ID
for file, id in re.findall(r'(.*): (.*)', data_txt_id):
    file_dict[file.strip()] = id.strip()

# Выводим словарь
file_dict

{'Постановление 1847.docx': '16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh',
 'СОГЛАШЕНИЕ Бурабай 2015.docx': '1_u0BBIJphrq2qBsqog5H0B9OgZjWO2bt',
 '162-ФЗ.docx': '1zQ_Xn771EuWl9-VZDAiQy2IThdpdgLys',
 'Приказ МПТ 971.docx': '15J5AZZvjRe_vCYze44tB-fk3amL-2YM0',
 'Приказ РСТ 2173.docx': '1r4wUVjTqWtNyB-rmI76DEsimSyBIqGlF',
 'Приказ РСТ 2346.docx': '1nJdos1CgVTcbtQtFhorMoxFPBx6BBgoe',
 'Постановление 521.docx': '1F17SNGzRYBiHNVLbB6DK6RlVx5LSbeRN',
 'ПМГ 06.2019.docx': '1CxtJO4dVKybY1swOkcXP4buJt1MYoBgb',
 'РМГ 29.2013.docx': '1qHfddurJ1QEc78sdAeHlVjBL3pflmnbJ',
 'Приказ МПТ 2907.docx': '1b30hYkfRiqO8TJVZpBwn5ui_RypiJxk0',
 'Приказ МПТ 456.docx': '1y_2d-GSEctG2q52Pu46fZ-L0cBppAL_y',
 'Приказ МПТ 4091.docx': '1R5pq1BhN59DXgx2MK-7sH0QbzJN_opWm',
 'Приказ МПТ 2167.docx': '1zuX2wQqoGKKl5eF9m1PuDTEqnQj9Z-1p',
 'Приказ МПТ 2906.docx': '1LCNEKwyZDQujUJgMHFdXL9x_kdbkwxoj',
 'Постановление 734.docx': '1qCFfa--cQb4crDu2uSCytUQcxsx7LlyJ',
 'Постановление 1053.docx': '1C0lbveUI49b9DVJWzaeTXvTkRFpyssfn',
 'Постано

In [7]:
# функция для загрузки документа по docx_id из гугл драйв в формате docx
def read_docx_from_id(docx_id):
    response = requests.get(f'https://docs.google.com/uc?export=download&id={docx_id}')
    file = io.BytesIO(response.content)
    document = Document(file)
    return document


In [8]:
#загрузим файл 'Постановление 1847.docx': '16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh'
doc = read_docx_from_id('16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh')
doc

In [9]:
#Смотрим основные свойства  документа
properties = doc.core_properties
print('Автор документа:', properties.author)
print('Автор последней правки:', properties.last_modified_by)
print('Дата создания документа:', properties.created)
print('Дата последней правки:', properties.modified)
print('Дата последней печати:', properties.last_printed)
print('Количество сохранений:', properties.revision)

Автор документа: Алексей Николаевич Крошкин
Автор последней правки: Алексей Николаевич Крошкин
Дата создания документа: 2024-04-11 08:45:00
Дата последней правки: 2024-04-11 08:45:00
Дата последней печати: None
Количество сохранений: 1


In [ ]:
#получаем весь текст из документа для дальнейшей обработки.
text = []
for paragraph in doc.paragraphs:
    text.append(paragraph.text)
print('\n'.join(text))


In [10]:
#Проходим через все абзацы в документе, собирает имена их стилей в множество
def print_paragraph_styles(document):
    styles = set()
    for paragraph in document.paragraphs:
        styles.add(paragraph.style.name)
    print("Paragraph styles:")
    for style in styles:
        print(style)

print_paragraph_styles(doc)

Paragraph styles:
Normal
ConsPlusTitlePage
ConsPlusNormal
ConsPlusTitle


In [12]:
def print_paragraph_style(document):
    styles = set()
    for paragraph in document.paragraphs:
        styles.add(paragraph.style.name)
    return styles

In [13]:
#Пройдемся по всем документам и сохраним применяемые стили
style_dict = {}
for file_name, file_id in file_dict.items():
    doc = read_docx_from_id(file_id)  # открываем документ
    style = print_paragraph_style(doc)  # применяем функцию к документу
    style_dict[file_name] = style

In [14]:
for file_name, style in style_dict.items():
  print (file_name, style)


Постановление 1847.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
СОГЛАШЕНИЕ Бурабай 2015.docx {'Normal'}
162-ФЗ.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 971.docx {'ConsPlusNormal', 'ConsPlusNonformat', 'Normal', 'ConsPlusTitlePage', 'ConsPlusTitle'}
Приказ РСТ 2173.docx {'ConsPlusNormal', 'ConsPlusNonformat', 'Normal', 'ConsPlusTitlePage', 'ConsPlusTitle'}
Приказ РСТ 2346.docx {'ConsPlusNormal', 'ConsPlusNonformat', 'Normal', 'ConsPlusTitlePage', 'ConsPlusTitle'}
Постановление 521.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
ПМГ 06.2019.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
РМГ 29.2013.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 2907.docx {'Normal', 'ConsPlusTitlePage', 'ConsPlusNormal', 'ConsPlusTitle'}
Приказ МПТ 456.docx {'ConsPlusNormal', 'ConsPlusNonformat', 'Normal', 'ConsPlusTitlePage', 'ConsPlusTitle'}
Приказ МПТ

Почти в каждом нормативном документе применяются одинаковые стили: ConsPlusTitlePage', 'ConsPlusTitle', 'Normal', 'ConsPlusNormal

In [ ]:
#проходит через каждый параграф и выводит его свойства, включая текст, выравнивание, стиль, шрифт, размер шрифта и форматирование (жирный, курсив, подчеркнутый).
def print_paragraphs_properties(doc):
    for i, paragraph in enumerate(doc.paragraphs):
        print(f"Параграф {i+1}:")
        print(f"Текст: {paragraph.text}")
        print(f"Выравнивание: {paragraph.alignment}")
        print(f"Стиль: {paragraph.style}")
        for run in paragraph.runs:
            print(f"Шрифт: {run.font.name}")
            print(f"Размер шрифта: {run.font.size}")
            print(f"Жирный: {run.bold}")
            print(f"Курсив: {run.italic}")
            print(f"Подчеркнутый: {run.underline}")
        print("\n")

print_paragraphs_properties(doc)

Стандартными средствами выводятся только стили параграфов

In [47]:
#Функция для преобразования документов docx, выгруженных из системы "КонсультантПлюс" в формат Markdown.
def preprocess_res_dub(document):
    # Создаем пустой список для строк Markdown
    markdown_lines = []
    # Инициализируем переменные для текущего абзаца и уровня заголовка
    current_paragraph = ""
    current_header_level = None
    # Проходим по каждому абзацу в документе
    for paragraph in document.paragraphs:
        # Получаем текст абзаца
        markdown_text = paragraph.text
        # Если текст абзаца не пустой
        if markdown_text:
            # Получаем XML структуру абзаца
            p_xml = paragraph._p.xml
            # Парсим XML структуру
            root = parse_xml(r'{}'.format(p_xml))
            # Определяем пространство имен для XML структуры
            namespaces = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
            # Ищем стиль абзаца в XML структуре
            style = root.find('.//w:pPr/w:pStyle', namespaces=namespaces)
            # Ищем уровень заголовка в XML структуре
            outline_lvl = root.find('.//w:pPr/w:outlineLvl', namespaces=namespaces)
            # Если стиль абзаца найден
            if style is not None:
                # Получаем значение стиля
                style_val = style.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')
                # Если стиль абзаца - это заголовок
                if style_val == 'ConsPlusTitle':
                    # Если уровень заголовка определен
                    if outline_lvl is not None:
                        # Получаем значение уровня заголовка и увеличиваем его на 1
                        outline_lvl_val = int(outline_lvl.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val'))
                        current_header_level = outline_lvl_val + 1
                    else:
                        # Если уровень заголовка не определен, устанавливаем его равным 1
                        current_header_level = 1
                    # Добавляем строку с заголовком в список строк Markdown
                    markdown_lines.append("#" * current_header_level + " " + markdown_text + "\n")
                    markdown_lines.append("#" * current_header_level + " " + markdown_text + "\n")
                else:
                    # Если абзац не является заголовком, просто добавляем его в список строк Markdown
                    markdown_lines.append(markdown_text + "\n")
    # Возвращаем список строк Markdown
    return markdown_lines

In [48]:
#Тест кода: загрузим файл 'Постановление 1847.docx': '16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh'
doc = read_docx_from_id('16XfjaU5Nie7NldFiDQGgPHcFrzqlVJKh')
preprocess_res_dub(doc)

['Документ предоставлен КонсультантПлюс\n\n',
 '# ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ\n',
 '# ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ\n',
 '# ПОСТАНОВЛЕНИЕ\n',
 '# ПОСТАНОВЛЕНИЕ\n',
 '# от 16 ноября 2020 г. N 1847\n',
 '# от 16 ноября 2020 г. N 1847\n',
 '# ОБ УТВЕРЖДЕНИИ ПЕРЕЧНЯ\n',
 '# ОБ УТВЕРЖДЕНИИ ПЕРЕЧНЯ\n',
 '# ИЗМЕРЕНИЙ, ОТНОСЯЩИХСЯ К СФЕРЕ ГОСУДАРСТВЕННОГО\n',
 '# ИЗМЕРЕНИЙ, ОТНОСЯЩИХСЯ К СФЕРЕ ГОСУДАРСТВЕННОГО\n',
 '# РЕГУЛИРОВАНИЯ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ\n',
 '# РЕГУЛИРОВАНИЯ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ\n',
 'В соответствии с частью 5 статьи 5 Федерального закона "Об обеспечении единства измерений" Правительство Российской Федерации постановляет:\n',
 '1. Утвердить прилагаемый перечень измерений, относящихся к сфере государственного регулирования обеспечения единства измерений, согласно приложению.\n',
 '2. Установить, что актуализация перечня, утвержденного настоящим постановлением, осуществляется на основании предложений Министерства промышленности и торговли Российско

In [43]:
#объединяем в один документ
doc_join = ''.join(preprocess_res_dub(doc))
doc_join

'Документ предоставлен КонсультантПлюс\n\nПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ\n# ПОСТАНОВЛЕНИЕ от 16 ноября 2020 г. N 1847 ОБ УТВЕРЖДЕНИИ ПЕРЕЧНЯ ИЗМЕРЕНИЙ, ОТНОСЯЩИХСЯ К СФЕРЕ ГОСУДАРСТВЕННОГО РЕГУЛИРОВАНИЯ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ \n# ПОСТАНОВЛЕНИЕ от 16 ноября 2020 г. N 1847 ОБ УТВЕРЖДЕНИИ ПЕРЕЧНЯ ИЗМЕРЕНИЙ, ОТНОСЯЩИХСЯ К СФЕРЕ ГОСУДАРСТВЕННОГО РЕГУЛИРОВАНИЯ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ \nВ соответствии с частью 5 статьи 5 Федерального закона "Об обеспечении единства измерений" Правительство Российской Федерации постановляет:\n1. Утвердить прилагаемый перечень измерений, относящихся к сфере государственного регулирования обеспечения единства измерений, согласно приложению.\n2. Установить, что актуализация перечня, утвержденного настоящим постановлением, осуществляется на основании предложений Министерства промышленности и торговли Российской Федерации, подготовленных совместно с заинтересованными федеральными органами исполнительной власти.\n3. Настоящее постановление вступае

In [49]:
#Еще один тест: 'ГОСТ 17025.docx': '1TDLp5FHonqT_t9jZCh-YG2nyYi-Fs29O'
doc = read_docx_from_id('1TDLp5FHonqT_t9jZCh-YG2nyYi-Fs29O')
preprocess_res_dub(doc)

['Документ предоставлен КонсультантПлюс\n\n',
 '# МЕЖГОСУДАРСТВЕННЫЙ СОВЕТ ПО СТАНДАРТИЗАЦИИ, МЕТРОЛОГИИ\n',
 '# МЕЖГОСУДАРСТВЕННЫЙ СОВЕТ ПО СТАНДАРТИЗАЦИИ, МЕТРОЛОГИИ\n',
 '# И СЕРТИФИКАЦИИ\n',
 '# И СЕРТИФИКАЦИИ\n',
 '# INTERSTATE COUNCIL FOR STANDARDIZATION,\n',
 '# INTERSTATE COUNCIL FOR STANDARDIZATION,\n',
 '# METROLOGY AND CERTIFICATION\n',
 '# METROLOGY AND CERTIFICATION\n',
 '# МЕЖГОСУДАРСТВЕННЫЙ СТАНДАРТ\n',
 '# МЕЖГОСУДАРСТВЕННЫЙ СТАНДАРТ\n',
 '# ГОСТ ISO/IEC 17025-2019\n',
 '# ГОСТ ISO/IEC 17025-2019\n',
 '# ОБЩИЕ ТРЕБОВАНИЯ\n',
 '# ОБЩИЕ ТРЕБОВАНИЯ\n',
 '# К КОМПЕТЕНТНОСТИ ИСПЫТАТЕЛЬНЫХ И КАЛИБРОВОЧНЫХ ЛАБОРАТОРИЙ\n',
 '# К КОМПЕТЕНТНОСТИ ИСПЫТАТЕЛЬНЫХ И КАЛИБРОВОЧНЫХ ЛАБОРАТОРИЙ\n',
 '# (ISO/IEC 17025:2017, IDT)\n',
 '# (ISO/IEC 17025:2017, IDT)\n',
 '# General requirements for the competence\n',
 '# General requirements for the competence\n',
 '# of testing and calibration laboratories\n',
 '# of testing and calibration laboratories\n',
 'Дата введения - 2019-09-01\n',
 

In [ ]:
#Код код предназначен для преобразования документа docx, выгруженных из системы "КонсультантПлюс" в формат Markdown. Он проходит через каждый абзац документа, анализирует его структуру XML и определяет уровень заголовка для каждого абзаца.
#Затем он добавляет соответствующее количество символов "#" перед каждым заголовком в соответствии с его уровнем. Если абзац не является заголовком, он просто добавляется в список строк Markdown.
#Тесты кода показывают, что он выделяет заголовки стиля ConsPlusTitle и маркирует от уровня заголовка.
#Однако, в нормативной документации есть документ СОГЛАШЕНИЕ Бурабай 2015.docx который не выгружен из КонсультантПлюс и не содержит структуры, для него нужно писать отдельную функцию(